In [4]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
MODEL_DIR = "/content/drive/MyDrive/phishing_ai/distilbert_model"

import os

print("Model exists:", os.path.exists(MODEL_DIR))
print("Files:")

for file in os.listdir(MODEL_DIR):
    print("-", file)

Model exists: True
Files:


In [5]:
from datasets import load_dataset

dataset = load_dataset("drorrabin/phishing_emails-data")

print(dataset)

README.md:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.88MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26946 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3705 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})


In [6]:
from collections import Counter

print("Labels:", dataset["train"].unique("email_type"))
print("Class distribution:", Counter(dataset["train"]["email_type"]))
print("Sample email:")
print(dataset["train"][0])

Labels: ['phishing email', 'safe email']
Class distribution: Counter({'phishing email': 13473, 'safe email': 13473})
Sample email:
{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.com/video/partners/email/index.html?url/video/crime/2008/08\n\nEmail type is: phishing email', '

In [7]:
def add_label(example):
    if example["email_type"] == "safe email":
        example["label"] = 0
    else:
        example["label"] = 1
    return example

dataset = dataset.map(add_label)

print("Train columns:", dataset["train"].column_names)
print("Label values:", dataset["train"].unique("label"))
print("Label distribution:", Counter(dataset["train"]["label"]))

Map:   0%|          | 0/26946 [00:00<?, ? examples/s]

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Train columns: ['text', 'email_type', 'label']
Label values: [1, 0]
Label distribution: Counter({1: 13473, 0: 13473})


In [9]:
from datasets import ClassLabel

dataset = dataset.cast_column(
    "label",
    ClassLabel(names=["SAFE", "PHISHING"])
)

print(dataset["train"].features)

Casting the dataset:   0%|          | 0/26946 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3705 [00:00<?, ? examples/s]

{'text': Value('string'), 'email_type': Value('string'), 'label': ClassLabel(names=['SAFE', 'PHISHING'])}


In [10]:
split_dataset = dataset["train"].train_test_split(
    test_size=0.10,
    seed=42,
    stratify_by_column="label"
)

train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training label distribution:", Counter(train_dataset["label"]))
print("Validation label distribution:", Counter(val_dataset["label"]))

Training samples: 24251
Validation samples: 2695
Training label distribution: Counter({0: 12126, 1: 12125})
Validation label distribution: Counter({1: 1348, 0: 1347})


In [11]:
def clean_email(example):
    example["clean_text"] = example["text"]
    return example

train_dataset = train_dataset.map(clean_email)
val_dataset = val_dataset.map(clean_email)

print("Training columns:", train_dataset.column_names)
print("Validation columns:", val_dataset.column_names)
print("Sample text:")
print(train_dataset[0]["clean_text"][:300])

Map:   0%|          | 0/24251 [00:00<?, ? examples/s]

Map:   0%|          | 0/2695 [00:00<?, ? examples/s]

Training columns: ['text', 'email_type', 'label', 'clean_text']
Validation columns: ['text', 'email_type', 'label', 'clean_text']
Sample text:
Is the following email safe or phishing??

Date: Thu, 07 Aug 2008 12:36:14 -0700

Sender: Adobe Edge <direct@adobesystems-macromedia.com>

Receiver: sigad@gvc.ceas-challenge.cc

Email Subject: Your Adobe Subscription Confirmation

Email Body: ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [12]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)
print("Maximum length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded: distilbert-base-uncased
Maximum length: 512


In [13]:
def chunk_email(example):
    tokenized = tokenizer(
        example["clean_text"],
        max_length=512,
        truncation=True,
        stride=64,
        return_overflowing_tokens=True,
        padding=False
    )

    result = {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "label": [example["label"]] * len(tokenized["input_ids"])
    }

    return result

print("Chunking function created successfully")

Chunking function created successfully


In [14]:
test_chunks = chunk_email(train_dataset[0])

print("Number of chunks:", len(test_chunks["input_ids"]))
print("First chunk tokens:", len(test_chunks["input_ids"][0]))
print("Labels:", test_chunks["label"])

Number of chunks: 1
First chunk tokens: 426
Labels: [0]


In [15]:
chunked_train = train_dataset.map(
    chunk_email,
    batched=True,
    batch_size=100,
    remove_columns=train_dataset.column_names
)

chunked_val = val_dataset.map(
    chunk_email,
    batched=True,
    batch_size=100,
    remove_columns=val_dataset.column_names
)

print("Training chunks:", len(chunked_train))
print("Validation chunks:", len(chunked_val))
print("Training columns:", chunked_train.column_names)
print("Validation columns:", chunked_val.column_names)

Map:   0%|          | 0/24251 [00:00<?, ? examples/s]

Map:   0%|          | 0/2695 [00:00<?, ? examples/s]

Training chunks: 24295
Validation chunks: 2700
Training columns: ['label', 'input_ids', 'attention_mask']
Validation columns: ['label', 'input_ids', 'attention_mask']


In [16]:
#load DistilBERT
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={
        0: "SAFE",
        1: "PHISHING"
    },
    label2id={
        "SAFE": 0,
        "PHISHING": 1
    }
)

model = model.to("cuda")

print("Model loaded successfully")
print("Device:", model.device)
print("Labels:", model.config.id2label)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully
Device: cuda:0
Labels: {0: 'SAFE', 1: 'PHISHING'}


In [17]:
#training config
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_DIR,

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,

    report_to="none"
)

print("Training configuration created")
print("Output directory:", training_args.output_dir)
print("Epochs:", training_args.num_train_epochs)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)

Training configuration created
Output directory: /content/drive/MyDrive/phishing_ai/distilbert_model
Epochs: 2
Train batch size: 8
Learning rate: 2e-05


In [18]:
#metric function
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predicted_labels = np.argmax(predictions, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predicted_labels,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predicted_labels)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Metrics function created successfully")

Metrics function created successfully


In [19]:
#dynamic padding collator
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator created successfully")

Data collator created successfully


In [20]:
#create Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=chunked_train,
    eval_dataset=chunked_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created successfully")
print("Training chunks:", len(chunked_train))
print("Validation chunks:", len(chunked_val))

Trainer created successfully
Training chunks: 24295
Validation chunks: 2700


In [21]:
batch = data_collator([
    chunked_train[i]
    for i in range(4)
])

print("Batch keys:", batch.keys())
print("Input shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels:", batch["labels"])

Batch keys: KeysView({'input_ids': tensor([[  101,  2003,  1996,  ...,  3647, 10373,   102],
        [  101,  2003,  1996,  ...,     0,     0,     0],
        [  101,  2003,  1996,  ...,     0,     0,     0],
        [  101,  2003,  1996,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0,
         0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0,
         1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0,
         0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1,
         1, 1, 0, 1],
        [0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0,
         0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0,
         1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 

In [23]:
def chunk_emails(batch):
    tokenized = tokenizer(
        batch["clean_text"],
        max_length=512,
        truncation=True,
        stride=64,
        return_overflowing_tokens=True,
        padding=False
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")

    tokenized["label"] = [
        batch["label"][sample_idx]
        for sample_idx in sample_mapping
    ]

    return tokenized

print("Batched chunking function created successfully")

Batched chunking function created successfully


In [25]:
#recreate chunked datasets
chunked_train = train_dataset.map(
    chunk_emails,
    batched=True,
    batch_size=100,
    remove_columns=train_dataset.column_names
)

chunked_val = val_dataset.map(
    chunk_emails,
    batched=True,
    batch_size=100,
    remove_columns=val_dataset.column_names
)

print("Training chunks:", len(chunked_train))
print("Validation chunks:", len(chunked_val))
print("Training columns:", chunked_train.column_names)
print("Validation columns:", chunked_val.column_names)

Map:   0%|          | 0/24251 [00:00<?, ? examples/s]

Map:   0%|          | 0/2695 [00:00<?, ? examples/s]

Training chunks: 24295
Validation chunks: 2700
Training columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Validation columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [26]:
batch = data_collator([
    chunked_train[i]
    for i in range(8)
])

print("Input shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)
print("Number of labels:", len(batch["labels"]))

Input shape: torch.Size([8, 426])
Attention mask shape: torch.Size([8, 426])
Labels shape: torch.Size([8])
Number of labels: 8


In [27]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=chunked_train,
    eval_dataset=chunked_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer recreated successfully")
print("Training chunks:", len(chunked_train))
print("Validation chunks:", len(chunked_val))

Trainer recreated successfully
Training chunks: 24295
Validation chunks: 2700


In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000008,0.000004,1.000000,1.000000,1.000000,1.000000


OSError: [Errno 30] Read-only file system: '/content/drive/MyDrive/phishing_ai/distilbert_model/checkpoint-3037'

In [29]:
LOCAL_MODEL_DIR = "/content/distilbert_model_final"

import os
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

trainer.save_model(LOCAL_MODEL_DIR)
tokenizer.save_pretrained(LOCAL_MODEL_DIR)

print("Local model saved!")
print("Files:")

for file in os.listdir(LOCAL_MODEL_DIR):
    print("-", file)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Local model saved!
Files:
- tokenizer_config.json
- model.safetensors
- training_args.bin
- config.json
- tokenizer.json


In [30]:
training_args.save_strategy = "no"
training_args.load_best_model_at_end = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=chunked_train,
    eval_dataset=chunked_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer recreated without checkpoint saving")

Trainer recreated without checkpoint saving


In [31]:
training_args.num_train_epochs = 1

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=chunked_train,
    eval_dataset=chunked_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready for final epoch")
print("Additional epochs:", training_args.num_train_epochs)

Trainer ready for final epoch
Additional epochs: 1


In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=3037, training_loss=5.0547172413242855e-08, metrics={'train_runtime': 226.8968, 'train_samples_per_second': 107.075, 'train_steps_per_second': 13.385, 'total_flos': 2016989231822172.0, 'train_loss': 5.0547172413242855e-08, 'epoch': 1.0})

In [33]:
FINAL_MODEL_DIR = "/content/distilbert_model_final"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("FINAL MODEL SAVED")
print("Files:")

import os

for file in os.listdir(FINAL_MODEL_DIR):
    print("-", file)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FINAL MODEL SAVED
Files:
- tokenizer_config.json
- model.safetensors
- training_args.bin
- config.json
- tokenizer.json


In [35]:
from google.colab import drive

drive.flush_and_unmount()

drive.mount("/content/drive", force_remount=True)

print("Drive remounted successfully")

Mounted at /content/drive
Drive remounted successfully


In [37]:
import shutil

zip_path = shutil.make_archive(
    "/content/distilbert_model_final",
    "zip",
    "/content/distilbert_model_final"
)

print("ZIP created:", zip_path)

ZIP created: /content/distilbert_model_final.zip


In [39]:
from google.colab import files

files.download("/content/distilbert_model_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
# Prepare the untouched test set

def prepare_test_label(example):
    label_map = {
        "safe email": 0,
        "phishing email": 1
    }

    return {
        "clean_text": example["text"],
        "label": label_map[example["email_type"]]
    }

test_dataset = dataset["test"].map(prepare_test_label)

print("Test emails:", len(test_dataset))
print("Test columns:", test_dataset.column_names)
print("Label distribution:", Counter(test_dataset["label"]))

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Test emails: 3705
Test columns: ['text', 'email_type', 'label', 'clean_text']
Label distribution: Counter({0: 3369, 1: 336})


In [42]:
chunked_test = test_dataset.map(
    chunk_emails,
    batched=True,
    batch_size=100,
    remove_columns=test_dataset.column_names
)

print("Test chunks:", len(chunked_test))
print("Test columns:", chunked_test.column_names)

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Test chunks: 3720
Test columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [43]:
test_results = trainer.evaluate(
    eval_dataset=chunked_test
)

print("Test Results:")
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000000,0.129948,1,0.983333,0.900585,0.916667,0.908555


Test Results:
{'eval_loss': 0.12994776666164398, 'eval_accuracy': 0.9833333333333333, 'eval_precision': 0.9005847953216374, 'eval_recall': 0.9166666666666666, 'eval_f1': 0.9085545722713865}


In [44]:
#confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

predictions = trainer.predict(chunked_test)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

actual_labels = predictions.label_ids

cm = confusion_matrix(actual_labels, predicted_labels)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        actual_labels,
        predicted_labels,
        target_names=["SAFE", "PHISHING"],
        digits=4
    )
)

Confusion Matrix:
[[3350   34]
 [  28  308]]

Classification Report:
              precision    recall  f1-score   support

        SAFE     0.9917    0.9900    0.9908      3384
    PHISHING     0.9006    0.9167    0.9086       336

    accuracy                         0.9833      3720
   macro avg     0.9461    0.9533    0.9497      3720
weighted avg     0.9835    0.9833    0.9834      3720



In [52]:
def predict_email(text):

    encoded = tokenizer(
        text,
        max_length=512,
        truncation=True,
        stride=64,
        return_overflowing_tokens=True,
        padding=False
    )

    batch = tokenizer.pad(
        {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"]
        },
        padding=True,
        return_tensors="pt"
    )

    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    probabilities = torch.softmax(outputs.logits, dim=-1)

    # Use the highest phishing probability among all chunks
    phishing_probability = probabilities[:, 1].max().item()
    safe_probability = 1 - phishing_probability

    if phishing_probability >= 0.50:
        predicted_label = 1
        confidence = phishing_probability
    else:
        predicted_label = 0
        confidence = safe_probability

    label_names = {
        0: "SAFE",
        1: "PHISHING"
    }

    return {
        "prediction": label_names[predicted_label],
        "confidence": confidence,
        "num_chunks": len(input_ids),
        "safe_probability": safe_probability,
        "phishing_probability": phishing_probability
    }


print("Updated risk-based email prediction function created")

Updated risk-based email prediction function created


In [46]:
sample_phishing = next(
    test_dataset[i]
    for i in range(len(test_dataset))
    if test_dataset[i]["label"] == 1
)

result = predict_email(sample_phishing["clean_text"])

print("Prediction:", result["prediction"])
print("Confidence:", f'{result["confidence"] * 100:.2f}%')
print("Number of chunks:", result["num_chunks"])
print("Safe probability:", f'{result["safe_probability"] * 100:.2f}%')
print("Phishing probability:", f'{result["phishing_probability"] * 100:.2f}%')

Prediction: PHISHING
Confidence: 100.00%
Number of chunks: 1
Safe probability: 0.00%
Phishing probability: 100.00%


In [47]:
#Test safe email
sample_safe = next(
    test_dataset[i]
    for i in range(len(test_dataset))
    if test_dataset[i]["label"] == 0
)

result = predict_email(sample_safe["clean_text"])

print("Prediction:", result["prediction"])
print("Confidence:", f'{result["confidence"] * 100:.2f}%')
print("Number of chunks:", result["num_chunks"])
print("Safe probability:", f'{result["safe_probability"] * 100:.2f}%')
print("Phishing probability:", f'{result["phishing_probability"] * 100:.2f}%')

Prediction: SAFE
Confidence: 100.00%
Number of chunks: 1
Safe probability: 100.00%
Phishing probability: 0.00%


In [53]:
#Long email test
token_check = tokenizer(
    long_test_email,
    truncation=False,
    add_special_tokens=True
)

print("Actual tokens:", len(token_check["input_ids"]))
print("DistilBERT maximum:", tokenizer.model_max_length)

result = predict_email(long_test_email)

print("\nPrediction:", result["prediction"])
print("Confidence:", f'{result["confidence"] * 100:.2f}%')
print("Number of chunks:", result["num_chunks"])
print("Safe probability:", f'{result["safe_probability"] * 100:.2f}%')
print("Phishing probability:", f'{result["phishing_probability"] * 100:.2f}%')

Actual tokens: 627
DistilBERT maximum: 512

Prediction: PHISHING
Confidence: 94.76%
Number of chunks: 2
Safe probability: 5.24%
Phishing probability: 94.76%


In [51]:
encoded = tokenizer(
    long_test_email,
    max_length=512,
    truncation=True,
    stride=64,
    return_overflowing_tokens=True,
    padding=False
)

batch = tokenizer.pad(
    {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"]
    },
    padding=True,
    return_tensors="pt"
)

input_ids = batch["input_ids"].to(model.device)
attention_mask = batch["attention_mask"].to(model.device)

model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

probabilities = torch.softmax(outputs.logits, dim=-1)

for i, probs in enumerate(probabilities):
    print(f"Chunk {i+1}:")
    print(f"  SAFE:     {probs[0].item() * 100:.2f}%")
    print(f"  PHISHING: {probs[1].item() * 100:.2f}%")

Chunk 1:
  SAFE:     5.24%
  PHISHING: 94.76%
Chunk 2:
  SAFE:     98.22%
  PHISHING: 1.78%
